# Session 1 — Hands-On Lab Notebook
## LLM Architecture & Model Behavior

**Audience:** McKinsey Technical Professionals (Data Engineers, ML Engineers, Backend & Cloud Engineers)  
**Session Duration:** 90–110 minutes  
**Labs:** 4 exercises covering tokenization, attention probing, parameter tuning, and failure detection

---

### 📋 Lab Map

| Lab | Title | Learning Units | Est. Time |
|-----|-------|---------------|-----------|
| **Lab 1** | Tokenization & the Token Pipeline | S1-01 to 04, S1-21 | 25 min |
| **Lab 2** | Context Window & Attention Probing | S1-05 to 12 | 25 min |
| **Lab 3** | Sampling Parameter Tuning | S1-13 to 16, S1-22 | 25 min |
| **Lab 4** | Programmatic Failure Detection | S1-17 to 20 | 20 min |

---

> **How to use this notebook:**  
> Work through each lab sequentially. Every section has a **Goal**, a **Setup** block, numbered exercises, and **Reflection** prompts. Fill in your observations and answers directly in the notebook. Code cells marked `# YOUR CODE HERE` require you to complete them.


---
## ⚙️ Global Setup

Run this cell once before starting any lab.

In [ ]:
# Install required packages (run once)
# Uncomment the line below if packages are not already installed
# !pip install tiktoken openai numpy

import os
import json
import time
import uuid
import numpy as np
from datetime import datetime

# Configure your API key 
# Set your key here OR use an environment variable
# os.environ["OPENAI_API_KEY"] = "sk-..."   # <-- uncomment and fill in

from openai import OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Model settings 
CHAT_MODEL      = "gpt-4o"          # swap to any available chat model
EMBEDDING_MODEL = "text-embedding-3-small"

print("Setup complete. Client initialised.")
print(f"   Chat model      : {CHAT_MODEL}")
print(f"   Embedding model : {EMBEDDING_MODEL}")


---
---
# Lab 1 — Tokenization & the Token Pipeline
**Learning Units:** S1-01 to 04, S1-21 · **Estimated Time:** 25 minutes

---

### Goal
Build hands-on intuition for how LLMs process text by:
- Tokenizing inputs and interpreting token boundaries
- Observing how small text changes shift token counts
- Identifying whether domain-specific terms tokenize cleanly
- Computing embedding similarities across semantically different sentence pairs

---

> **Key concept from S1-02:** The transformer does not process characters or words — it maps input to **tokens**
> (subword units from a fixed vocabulary). Token boundaries determine what the model can and cannot compute reliably.


### 🔧 Lab 1 Setup

In [ ]:
import tiktoken

# Load the tokenizer for your chat model
enc = tiktoken.encoding_for_model("gpt-4o")

def tokenize(text, encoder=enc):
    """Return token IDs and their decoded string representations."""
    token_ids = encoder.encode(text)
    token_strings = [encoder.decode([t]) for t in token_ids]
    return token_ids, token_strings

def show_tokens(text, encoder=enc):
    """Pretty-print token breakdown for a given string."""
    ids, strings = tokenize(text, encoder)
    print(f"Input   : {repr(text)}")
    print(f"Count   : {len(ids)} token(s)")
    print(f"Tokens  : {strings}")
    print()

print("Tokenizer ready.")


---
### Exercise 1.1 — Tokenize and Count

**S1-01 / S1-02 connection:** LLM outputs are stochastic and architecturally different from classical ML. One key reason: the model operates on tokens, not text. Understanding token boundaries is the prerequisite for writing reliable prompts.


In [ ]:
# ── Step 1: Tokenize a baseline sentence ───────────────────────────────────
baseline = "The transformer attention mechanism scales quadratically with sequence length."
show_tokens(baseline)


In [ ]:
# ── Step 2: Vary the sentence and observe token count changes ───────────────
# Run each variation below and compare counts with the baseline

variations = {
    "With comma after 'transformer'":
        "The transformer, attention mechanism scales quadratically with sequence length.",

    "All words capitalised":
        "The Transformer Attention Mechanism Scales Quadratically With Sequence Length.",

    "Domain-specific term substituted":
        "The multi-head self-attention mechanism scales quadratically with sequence length.",
        # Replace with a term from YOUR domain

    "Translated to another language":
        "Der Transformer-Aufmerksamkeitsmechanismus skaliert quadratisch mit der Sequenzlänge.",
        # Replace with a language you work in or are curious about
}

print(f"{'Variation':<45} {'Tokens':>6}")
print("-" * 53)
baseline_ids, _ = tokenize(baseline)
print(f"{'[Baseline]':<45} {len(baseline_ids):>6}")

for label, text in variations.items():
    ids, _ = tokenize(text)
    print(f"{label:<45} {len(ids):>6}")


In [ ]:
# ── Step 3: Record your observations ──────────────────────────────────────
# Which variation changed the token count most? Why do you think that is?

observation = """
YOUR OBSERVATION HERE
(e.g., Capitalising each word added N tokens because...)
"""
print(observation)


---
### Exercise 1.2 — Domain Term Tokenization

**S1-03 connection:** If a term you use in prompts splits into multiple sub-tokens, the model is computing attention over fragments — not the whole term. This affects reliability for that term.


In [ ]:
# ── Step 1: Tokenize three domain-specific terms you use in prompts ─────────
domain_terms = [
    "clientEngagementID",    # replace with YOUR terms
    "YourDomainTerm2",
    "YourDomainTerm3",
]

print(f"{'Term':<30} {'Tokens':>6}  {'Breakdown'}")
print("-" * 70)
for term in domain_terms:
    ids, strings = tokenize(term)
    flag = " multi-token" if len(ids) > 1 else " single token"
    print(f"{term:<30} {len(ids):>6}  {strings}  {flag}")


In [ ]:
# ── Step 2: For any multi-token term, write the implication for your prompts ─
implications = """
Term: [fill in]
Implication: When I use this term in a prompt, the model processes it as N fragments.
This means I should consider... [fill in your reasoning]
"""
print(implications)


---
### Exercise 1.3 — Embedding Similarity

**S1-02 connection:** Each token maps to an embedding — a high-dimensional vector encoding semantic and syntactic relationships. Computing cosine similarity between sentence embeddings shows how the model's training distribution represents meaning.


In [ ]:
def get_embedding(text):
    """Return the embedding vector for a text string."""
    response = client.embeddings.create(input=text, model=EMBEDDING_MODEL)
    return np.array(response.data[0].embedding)

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(" Embedding helpers ready.")


In [ ]:
# ── Step 1: Define three sentence pairs ─────────────────────────────────────
pairs = {
    "Semantically similar": (
        "The model exceeded expectations.",
        "The AI system outperformed benchmarks.",
    ),
    "Syntactically similar, semantically different": (
        "The bank by the river flooded.",
        "The bank approved the loan application.",
    ),
    "Unrelated": (
        "The quarterly report is due on Friday.",
        "The transformer uses positional encoding to preserve word order.",
    ),
}

# ── Step 2: Compute and display similarity scores ───────────────────────────
print(f"{'Pair Type':<45} {'Similarity':>10}")
print("-" * 57)

for label, (s1, s2) in pairs.items():
    emb1 = get_embedding(s1)
    emb2 = get_embedding(s2)
    sim  = cosine_similarity(emb1, emb2)
    print(f"{label:<45} {sim:>10.4f}")


In [ ]:
# ── Step 3: Did any score surprise you? ─────────────────────────────────────
# Record results and any surprises below

surprising_result = """
Pair that surprised me: [fill in]
Score: [fill in]
Why I found it surprising / what it implies for prompting: [fill in]
"""
print(surprising_result)


---
### 💭 Lab 1 — Reflection

Answer these questions before moving to Lab 2.


In [ ]:
reflections_lab1 = {
    "Q1 - Token boundary implication":
        """[Your answer]
        Name one case in your current or recent work where knowing the token boundary
        of a key term would have changed how you wrote a prompt.""",

    "Q2 - Why LLMs can't count letters":
        """[Your answer using vocabulary from this lab]
        Why can an LLM not reliably count occurrences of the letter 'e' in a word?""",

    "Q3 - Schema key tokenization":
        """[Your answer]
        You are building a JSON extraction pipeline. Your schema key is 'clientEngagementID'.
        Should you test whether it tokenizes cleanly? Why or why not?""",
}

for q, a in reflections_lab1.items():
    print(f"{'='*60}")
    print(f"{q}")
    print(f"{a}")
    print()


---
---
# 🔬 Lab 2 — Context Window & Attention Probing
**Learning Units:** S1-05 to 12 · **Estimated Time:** 25 minutes

---

### 🎯 Goal
Empirically demonstrate and diagnose attention-related failure patterns:
- Run a controlled "lost-in-the-middle" experiment
- Distinguish **attention dilution** (context structure) from **instruction failure** (prompt quality)
- Apply context positioning strategy to improve retrieval accuracy

---

> **Key concept from S1-09:** Attention in a transformer is not a search function.
> Information buried in the middle of a long context is retrieved less reliably than
> information at the boundaries — this is the **lost-in-the-middle** effect.


### 🔧 Lab 2 Setup

In [ ]:
def ask_llm(system_prompt, user_message, temperature=0.0, model=CHAT_MODEL):
    """Send a single chat completion request and return the response string."""
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ]
    )
    return response.choices[0].message.content.strip()

# ── Prepare a long document ──────────────────────────────────────────────────
# Paste or generate a text document of at least 1,500 words.
# It must contain one specific, verifiable fact (a number, name, or date).
# We call the paragraph containing that fact the TARGET PARAGRAPH.

DOCUMENT_TEXT = """
[PASTE YOUR 1,500+ WORD DOCUMENT HERE]

Ensure this document contains at least one paragraph with a checkable fact,
e.g., "The project budget was approved at $4.7 million on March 12, 2023."
"""

TARGET_FACT_QUESTION = "What was the approved project budget and on what date?"
# Replace with a question whose answer is in your document

print(f"Document word count (approx): {len(DOCUMENT_TEXT.split())}")
print(f"Target question: {TARGET_FACT_QUESTION}")


---
### Exercise 2.1 — The Lost-in-the-Middle Experiment

**S1-05 / S1-09 connection:** Context window failures are not bugs in your code — they are architectural properties of the transformer. Attention quality degrades for information positioned in the middle of long inputs.


In [ ]:
def build_document_version(full_text, target_paragraph, position):
    """
    Return a version of the document with target_paragraph placed at
    'beginning', 'middle', or 'end'.
    
    Simple approach: split into thirds and insert target paragraph
    at the desired position boundary.
    """
    paragraphs = [p.strip() for p in full_text.strip().split("\n\n") if p.strip()]

    # Remove the target paragraph from wherever it currently sits
    remaining = [p for p in paragraphs if target_paragraph.strip() not in p]

    n = len(remaining)
    if position == "beginning":
        ordered = [target_paragraph] + remaining
    elif position == "middle":
        mid = n // 2
        ordered = remaining[:mid] + [target_paragraph] + remaining[mid:]
    else:  # end
        ordered = remaining + [target_paragraph]

    return "\n\n".join(ordered)


# ── Define your target paragraph ────────────────────────────────────────────
TARGET_PARAGRAPH = """
[PASTE THE PARAGRAPH CONTAINING YOUR TARGET FACT HERE]
"""

SYSTEM = "Answer only from the provided document. Be specific and concise."

versions = {
    "beginning": build_document_version(DOCUMENT_TEXT, TARGET_PARAGRAPH, "beginning"),
    "middle":    build_document_version(DOCUMENT_TEXT, TARGET_PARAGRAPH, "middle"),
    "end":       build_document_version(DOCUMENT_TEXT, TARGET_PARAGRAPH, "end"),
}

print(" Document versions prepared.")
for name, doc in versions.items():
    print(f"  {name}: ~{len(doc.split())} words")


In [ ]:
# ── Run 3 trials per version and record correctness ─────────────────────────
# After each run, manually mark True if the model retrieved the target fact correctly.

results = {}

for position, doc_text in versions.items():
    user_msg = f"{doc_text}\n\n---\nQuestion: {TARGET_FACT_QUESTION}"
    trial_results = []
    
    for trial in range(1, 4):
        answer = ask_llm(SYSTEM, user_msg)
        # Print for manual review
        print(f"[{position.upper()} | Trial {trial}]")
        print(f"  Answer: {answer[:200]}...")
        correct = input("  Was this correct? (y/n): ").strip().lower() == "y"
        trial_results.append(correct)
        print()
    
    correct_count = sum(trial_results)
    results[position] = correct_count
    print(f" {position}: {correct_count}/3 correct\n")

print("\n── Summary ──────────────────────────────────────────")
for pos, score in results.items():
    bar = "#" * score + "~" * (3 - score)
    print(f"  {pos:<12}: {bar}  ({score}/3)")


In [ ]:
# ── Non-interactive version (if running headless / in batch) ────────────────
# Uncomment and fill in your manually observed results here:

# results_manual = {
#     "beginning": 3,   # fill in your observed correct count out of 3
#     "middle":    1,
#     "end":       3,
# }
# 
# print("Manual results:")
# for pos, score in results_manual.items():
#     bar = "#" * score + "~" * (3 - score)
#     print(f"  {pos:<12}: {bar}  ({score}/3)")


---
### Exercise 2.2 — Attention Dilution vs. Instruction Failure

**S1-11 connection:** When a model ignores relevant information, the diagnostic question is not "what is wrong with my instruction?" but "how is attention distributed across this input?" These two root causes require different fixes.


In [ ]:
# ── Step 1: Reduced Input Test ──────────────────────────────────────────────
# Send ONLY the target paragraph with the same question.
# If the model answers correctly here but not in the full-doc test - ATTENTION DILUTION
# If the model fails here too - INSTRUCTION / PASSAGE FAILURE

reduced_user_msg = f"{TARGET_PARAGRAPH}\n\n---\nQuestion: {TARGET_FACT_QUESTION}"
reduced_answer = ask_llm(SYSTEM, reduced_user_msg)

print("── Reduced Input Result ─────────────────────────────────────────────")
print(reduced_answer)


In [ ]:
# ── Step 2: Diagnose ────────────────────────────────────────────────────────
# Based on your observations from Exercises 2.1 and 2.2 (Step 1), 
# fill in your diagnosis below.

diagnosis = "ATTENTION DILUTION"   # or "INSTRUCTION FAILURE"

reasoning = """
[Explain your reasoning]
e.g., The model answered correctly on the reduced input but failed on Version B (middle position),
confirming that the relevant passage was present in the context but underweighted by attention.
"""

print(f"Diagnosis: {diagnosis}")
print(f"Reasoning: {reasoning}")


In [ ]:
# ── Step 3 (if ATTENTION DILUTION): Reposition and retest ──────────────────
# Move the target paragraph to the BEGINNING and run 3 more trials.

repositioned_doc = build_document_version(DOCUMENT_TEXT, TARGET_PARAGRAPH, "beginning")
repositioned_msg = f"{repositioned_doc}\n\n---\nQuestion: {TARGET_FACT_QUESTION}"

print("── Repositioned (beginning) — running 3 trials ─────────────────────")
for trial in range(1, 4):
    answer = ask_llm(SYSTEM, repositioned_msg)
    print(f"  Trial {trial}: {answer[:200]}...")
    print()

# Record your new correct count:
repositioned_correct = None  # fill in: e.g. 3
print(f"\nNew retrieval count after repositioning: {repositioned_correct} / 3")


---
### 💭 Lab 2 — Reflection

In [ ]:
reflections_lab2 = {
    "Q1 - Context positioning strategy":
        """[Your answer — 2 sentences]
        You are building a contract review tool that sends 40-page contracts to an LLM.
        Where would you position critical clauses in the context? Describe your strategy.""",

    "Q2 - Reframing 'not reading carefully'":
        """[Your answer]
        A colleague says the model is 'not reading carefully enough.'
        Is that an accurate description of the failure? What would you say instead?""",

    "Q3 - Your use case context budget":
        """[Your answer]
        Name one use case from your current work where context window management
        would be the binding constraint. What is the approximate token budget?""",
}

for q, a in reflections_lab2.items():
    print(f"{'='*60}")
    print(f"{q}")
    print(f"{a}")
    print()


---
---
# 🔬 Lab 3 — Sampling Parameter Tuning
**Learning Units:** S1-13 to 16, S1-22 · **Estimated Time:** 25 minutes

---

### 🎯 Goal
Develop calibrated intuition for temperature and top-p effects by:
- Running a temperature sweep on a structured JSON extraction task
- Identifying the compliance threshold temperature for your use case
- Testing top-p combinations and documenting a defensible parameter configuration

---

> **Key concept from S1-14:** Temperature and top-p are not style preferences —
> they are **probability distribution controls**. Output variability, format instability,
> and length inconsistency are sampling parameter symptoms, not prompt quality symptoms.

---

### 📊 Parameter Reference

| Parameter | Low (0.0–0.3) | Mid (0.4–0.7) | High (0.8–1.2) |
|-----------|---------------|---------------|-----------------|
| **Temperature** | Deterministic. Best for JSON, code, structured fields. | Balanced. Summaries, analysis. | High variance. Creative generation only. |
| **Top-p** | Tight nucleus. Reduces unusual token selection. | Standard for most tasks. | Wide nucleus. More lexical diversity. |
| **Freq. Penalty** | No repetition suppression. | Light suppression for long prose. | Aggressive. Risk of incoherence above 0.7. |


### 🔧 Lab 3 Setup

In [ ]:
# ── Structured output prompt ─────────────────────────────────────────────────
EXTRACTION_SYSTEM = (
    "Extract the requested fields. Return ONLY valid JSON. "
    "No preamble, explanation, or markdown fences."
)

EXTRACTION_USER = """
From the following text, extract these fields:
- project_name   (string)
- budget_usd     (integer)
- due_date       (string, format YYYY-MM-DD)
- owner          (string)

Text:
The Orion Analytics initiative, led by Sarah Chen, has been allocated a budget of
$2,400,000 with a delivery deadline of September 30, 2025.
"""
# You can replace this with a paragraph from your own domain

REQUIRED_KEYS   = ["project_name", "budget_usd", "due_date", "owner"]
EXPECTED_TYPES  = {
    "project_name": str,
    "budget_usd":   int,
    "due_date":     str,
    "owner":        str,
}

def run_extraction(temperature, top_p=1.0):
    """Run the extraction prompt and return (raw_output, passed, error_msg)."""
    raw = ask_llm(EXTRACTION_SYSTEM, EXTRACTION_USER,
                  temperature=temperature)
    try:
        parsed = json.loads(raw)
        missing = [k for k in REQUIRED_KEYS if k not in parsed]
        if missing:
            return raw, False, f"Missing keys: {missing}"
        type_errors = [
            f"{k} should be {EXPECTED_TYPES[k].__name__}, got {type(parsed[k]).__name__}"
            for k in REQUIRED_KEYS if not isinstance(parsed[k], EXPECTED_TYPES[k])
        ]
        if type_errors:
            return raw, False, f"Type errors: {type_errors}"
        return raw, True, None
    except json.JSONDecodeError as e:
        return raw, False, f"JSON parse error: {e}"

print("Extraction helpers ready.")


---
### Exercise 3.1 — Temperature Sweep

**S1-15 connection:** Parameter tuning is a controlled experiment. Define the success criterion first, then vary one parameter at a time.


In [ ]:
# ── Run 3 trials at each temperature ────────────────────────────────────────
temperatures = [0.0, 0.2, 0.5, 0.8, 1.2]
sweep_results = {}

for temp in temperatures:
    trials = []
    for i in range(3):
        raw, passed, error = run_extraction(temperature=temp)
        trials.append({"passed": passed, "error": error, "raw": raw})
    
    pass_count = sum(t["passed"] for t in trials)
    sweep_results[temp] = trials
    
    status = "OK" * pass_count + "NOT_OK" * (3 - pass_count)
    first_error = next((t["error"] for t in trials if not t["passed"]), "—")
    print(f"  temp={temp:<5} {status}  ({pass_count}/3 pass)  first error: {first_error}")


In [ ]:
# ── Identify the compliance threshold ───────────────────────────────────────
threshold_temp = None  # Fill in after reviewing results above

for temp in temperatures:
    pass_count = sum(t["passed"] for t in sweep_results[temp])
    if pass_count < 3 and threshold_temp is None:
        threshold_temp = temp

print(f"\n Compliance threshold temperature: {threshold_temp}")
print("   (First temperature at which at least one trial failed)")


In [ ]:
# ── Inspect a failing output ─────────────────────────────────────────────────
# Find the first failure and print the raw output to understand the failure type.

for temp, trials in sweep_results.items():
    for i, trial in enumerate(trials):
        if not trial["passed"]:
            print(f"First failure at temp={temp}, trial {i+1}")
            print(f"Error type : {trial['error']}")
            print(f"Raw output :\n{trial['raw']}")
            break
    else:
        continue
    break


---
### Exercise 3.2 — Top-p Comparison

**S1-14 connection:** Top-p (nucleus sampling) and temperature work together. Top-p restricts which tokens are eligible; temperature controls the sharpness of the distribution within that nucleus.


In [ ]:
# ── Use the highest fully-compliant temperature from Exercise 3.1 ────────────
compliant_temp = 0.0  #  update this based on your sweep results

top_p_values = [0.85, 0.90, 0.95, 1.0]
top_p_results = {}

print(f"Testing top-p values at temperature={compliant_temp}\n")

for tp in top_p_values:
    trials = []
    for i in range(3):
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            temperature=compliant_temp,
            top_p=tp,
            messages=[
                {"role": "system", "content": EXTRACTION_SYSTEM},
                {"role": "user",   "content": EXTRACTION_USER},
            ]
        )
        raw = response.choices[0].message.content.strip()
        try:
            parsed = json.loads(raw)
            passed = all(k in parsed for k in REQUIRED_KEYS)
            trials.append({"passed": passed, "raw": raw})
        except json.JSONDecodeError:
            trials.append({"passed": False, "raw": raw})
    
    pass_count = sum(t["passed"] for t in trials)
    top_p_results[tp] = trials
    print(f"  top_p={tp}  {'OK'*pass_count}{'NOT_OK'*(3-pass_count)}  ({pass_count}/3 pass)")


In [ ]:
# ── Record your final recommended configuration ─────────────────────────────
final_config = {
    "task_type":              "JSON field extraction",
    "temperature":            0.0,    # update with your finding
    "top_p":                  1.0,    # update with your finding
    "frequency_penalty":      0.0,    # adjust only if repetition observed
    "observed_compliance_rate": "X/3 at all tested temperatures ≤ threshold",  # fill in
    "failure_mode_at_higher_temp": "[describe what breaks first — key missing / type error / parse error]",
    "notes":                  "[any edge cases or caveats observed]",
}

print("── Final Parameter Configuration Card ──────────────────────────────")
for k, v in final_config.items():
    print(f"  {k:<35}: {v}")

# Treat this as documentation you would commit to a codebase.


---
### 💭 Lab 3 — Reflection

In [ ]:
reflections_lab3 = {
    "Q1 - Diagnosing a teammate's issue":
        """[Your answer]
        A developer on your team says: 'I keep tweaking the prompt but the JSON keeps breaking.'
        What is the first question you would ask them, based on this lab?""",

    "Q2 - Document routing pipeline":
        """[Your answer with justification]
        For a pipeline that routes client documents to analysts based on an LLM classification label,
        what temperature would you use and why?""",
}

for q, a in reflections_lab3.items():
    print(f"{'='*60}")
    print(f"{q}")
    print(f"{a}")
    print()


---
---
# Lab 4 — Programmatic Failure Detection
**Learning Units:** S1-17 to 20 · **Estimated Time:** 20 minutes

---

### Goal
Build the inspection infrastructure needed to surface failure patterns at scale:
- Instrument LLM calls with structured logging
- Define and run automated quality checks
- Identify whether failure distribution is patterned or random
- Scope human review requirements by output risk category

---

> **Key concept from S1-19:** Debugging LLM output by reading individual responses is not scalable.
> Failure patterns emerge across **collections** of responses — hallucination rates, format compliance,
> reasoning errors — not from individual examples.


### 🔧 Lab 4 Setup

In [ ]:
# ── Shared log store (list of dicts) ─────────────────────────────────────────
LOG_STORE = []

print("Log store initialised.")


---
### Exercise 4.1 — Build a Structured Logger

**S1-19 connection:** Every LLM call in your system should log the full prompt, raw response, parameters, token counts, and a timestamp — in a format you can query.


In [ ]:
def logged_llm_call(system_prompt, user_message,
                    temperature=0.0, top_p=1.0,
                    model=CHAT_MODEL, log_store=LOG_STORE):
    """
    Send a chat completion request and append a structured log entry.
    Returns (raw_output, log_entry).
    """
    start_time = time.time()
    
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        top_p=top_p,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ]
    )
    
    latency_ms   = round((time.time() - start_time) * 1000, 1)
    raw_output   = response.choices[0].message.content.strip()
    usage        = response.usage

    log_entry = {
        "id":            str(uuid.uuid4()),
        "timestamp":     datetime.utcnow().isoformat(),
        "model":         model,
        "temperature":   temperature,
        "top_p":         top_p,
        "input_tokens":  usage.prompt_tokens,        #  TODO was here
        "output_tokens": usage.completion_tokens,     #  TODO was here
        "latency_ms":    latency_ms,                  #  TODO was here
        "system_prompt": system_prompt,
        "user_message":  user_message,
        "raw_output":    raw_output,
    }
    
    log_store.append(log_entry)
    return raw_output, log_entry


print(" Logged call wrapper ready.")


In [ ]:
# ── Run 3 calls through the logger ───────────────────────────────────────────
for i in range(3):
    raw, entry = logged_llm_call(
        system_prompt=EXTRACTION_SYSTEM,
        user_message=EXTRACTION_USER,
        temperature=0.0,
    )
    print(f"Call {i+1}: id={entry['id'][:8]}  "
          f"in={entry['input_tokens']} tok  "
          f"out={entry['output_tokens']} tok  "
          f"latency={entry['latency_ms']} ms")

print(f"\nTotal entries in LOG_STORE: {len(LOG_STORE)}")


In [ ]:
# ── Preview a log entry as formatted JSON ────────────────────────────────────
sample = LOG_STORE[-1].copy()
sample["system_prompt"] = sample["system_prompt"][:80] + "..."
sample["user_message"]  = sample["user_message"][:80]  + "..."
sample["raw_output"]    = sample["raw_output"][:80]     + "..."

print(json.dumps(sample, indent=2))


---
### Exercise 4.2 — Automated Quality Checks

**S1-19 connection:** Define automated checks for your output type. For structured outputs, parse every response against your schema and record whether it parses successfully. Track the failure rate over time.


In [ ]:
# ── Check 1: JSON schema validation ─────────────────────────────────────────
def check_json_schema(raw_output, required_keys=REQUIRED_KEYS,
                      expected_types=EXPECTED_TYPES):
    """
    Returns (passed: bool, error: str | None).
    Validates JSON parse, required key presence, and field types.
    """
    try:
        parsed = json.loads(raw_output)
    except json.JSONDecodeError as e:
        return False, f"JSON parse error: {e}"
    
    missing = [k for k in required_keys if k not in parsed]
    if missing:
        return False, f"Missing keys: {missing}"
    
    type_errors = [
        f"'{k}' expected {expected_types[k].__name__}, got {type(parsed[k]).__name__}"
        for k in required_keys
        if k in parsed and not isinstance(parsed[k], expected_types[k])
    ]
    if type_errors:
        return False, f"Type mismatch: {type_errors}"
    
    return True, None


# ── Check 2: Output token ceiling ───────────────────────────────────────────
# A structured extraction response should not require hundreds of tokens.
# Excessively long outputs often signal the model added unwanted explanation.

MAX_OUTPUT_TOKENS = 100

def check_output_token_ceiling(log_entry, max_tokens=MAX_OUTPUT_TOKENS):
    """Returns (passed, error)."""
    actual = log_entry["output_tokens"]
    if actual > max_tokens:
        return False, f"output_tokens={actual} exceeds ceiling of {max_tokens}"
    return True, None


# ── Check 3 (YOUR OWN): Add a third check below ─────────────────────────────
def check_no_preamble(raw_output):
    """
    Returns (passed, error).
    Verifies the model did not prepend explanation text before the JSON.
    Replace or extend this with your own check.
    """
    stripped = raw_output.strip()
    if not stripped.startswith("{"):
        return False, f"Output does not start with '{{' — possible preamble detected"
    return True, None


print("Quality check functions defined.")


In [ ]:
# ── Run all checks over the log store ────────────────────────────────────────
print(f"{'ID':<10} {'Schema':^8} {'Tokens':^8} {'No Preamble':^12} {'Overall'}")
print("-" * 52)

for entry in LOG_STORE:
    s_pass, s_err = check_json_schema(entry["raw_output"])
    t_pass, t_err = check_output_token_ceiling(entry)
    p_pass, p_err = check_no_preamble(entry["raw_output"])
    
    overall = "PASS" if all([s_pass, t_pass, p_pass]) else "FAIL"
    
    def fmt(p): return "PASS" if p else "FAIL"
    print(f"{entry['id'][:8]:<10} {fmt(s_pass):^8} {fmt(t_pass):^8} {fmt(p_pass):^12} {overall}")
    
    for err in [s_err, t_err, p_err]:
        if err:
            print(f"   {err}")


In [ ]:
# ── Now run calls at multiple temperatures to build a larger sample ──────────
# (Reuses logged_llm_call so results accumulate in LOG_STORE)

print("Running temperature sweep for failure pattern analysis...\n")

for temp in [0.0, 0.2, 0.5, 0.8, 1.2]:
    for _ in range(3):
        logged_llm_call(EXTRACTION_SYSTEM, EXTRACTION_USER,
                        temperature=temp)

print(f"Total log entries: {len(LOG_STORE)}")


---
### Exercise 4.3 — Failure Pattern Analysis

**S1-20 connection:** A pattern in failures is architectural or prompt-level evidence. A random distribution of failures suggests a sampling or temperature configuration issue.


In [ ]:
# ── Group failures by temperature ────────────────────────────────────────────
from collections import defaultdict

failures_by_temp = defaultdict(int)
totals_by_temp   = defaultdict(int)

for entry in LOG_STORE:
    temp  = entry["temperature"]
    passed, _ = check_json_schema(entry["raw_output"])
    totals_by_temp[temp] += 1
    if not passed:
        failures_by_temp[temp] += 1

print(f"{'Temperature':>12}  {'Failures':>8}  {'Total':>6}  {'Fail Rate':>9}  Distribution")
print("-" * 65)

for temp in sorted(totals_by_temp):
    total    = totals_by_temp[temp]
    failures = failures_by_temp[temp]
    rate     = failures / total if total else 0
    bar_f    = "#" * failures
    bar_p    = "~" * (total - failures)
    print(f"{temp:>12.1f}  {failures:>8}  {total:>6}  {rate:>8.0%}   {bar_f}{bar_p}")


In [ ]:
# ── Interpret the pattern ─────────────────────────────────────────────────────
pattern_interpretation = """
PATTERN TYPE: [CONCENTRATED AT HIGH TEMPERATURE  /  RANDOMLY DISTRIBUTED]

Evidence: [Describe what you observed — e.g., all failures at temp >= 0.8]

Implication:
  - Concentrated  sampling/parameter issue  fix: lower temperature
  - Random        prompt or schema issue    fix: revise instruction or output schema
"""

print(pattern_interpretation)


---
### Exercise 4.4 — Human Review Scoping

**S1-20 connection:** Human review is the mitigation for consequences you cannot afford to let reach a client. Scope it by output category and failure mode — not uniformly across the whole system.


In [ ]:
# ── Complete the human review scoping table ──────────────────────────────────
# Fill in the REVIEW_STRATEGY for each output category.

review_scope = [
    {
        "output_category":  "LLM classifies document  routes to automated workflow",
        "primary_failure":  "Schema / format failure propagates silently",
        "review_strategy":  "[YOUR ANSWER  e.g., schema validation + retry logic + sampled human review]",
    },
    {
        "output_category":  "LLM summarises findings delivered to client without independent verification",
        "primary_failure":  "[YOUR ANSWER  identify the primary failure mode]",
        "review_strategy":  "[YOUR ANSWER  describe the review approach]",
    },
    {
        "output_category":  "LLM generates chain-of-thought reasoning steps (internal, developer-reviewed)",
        "primary_failure":  "[YOUR ANSWER]",
        "review_strategy":  "[YOUR ANSWER]",
    },
]

print(f"{'Output Category':<52} {'Failure Mode':<30} {'Review Strategy'}")
print("=" * 120)
for row in review_scope:
    print(f"{row['output_category']:<52} {row['primary_failure']:<30} {row['review_strategy']}")
    print()


---
### 💭 Lab 4 — Reflection

In [ ]:
reflections_lab4 = {
    "Q1 - Plausible fake citation":
        """[Your answer]
        A model returns a plausible-sounding citation in a client output. You cannot immediately verify it.
        What is the correct next step — and why is adjusting the prompt NOT the primary fix?""",

    "Q2 - Patterned vs random failures":
        """[Your answer]
        You observe 12 failures over 100 production calls. 10 are on inputs longer than 800 tokens.
        Is this random or patterned? What does that tell you?""",

    "Q3 - Three mitigations in one sentence each":
        """
        (a) Hallucination mitigation    : [your one sentence]
        (b) Arithmetic error mitigation : [your one sentence]
        (c) Overconfident output mitigation: [your one sentence]
        """,
}

for q, a in reflections_lab4.items():
    print(f"{'='*60}")
    print(f"{q}")
    print(f"{a}")
    print()


---
## Take-Home Challenge

Identify one LLM integration you are currently building or have recently built.  
Using the frameworks from this session, write a short technical brief covering:

1. **Context window strategy** — how you manage input length and position critical information
2. **Parameter configuration rationale** — temperature, top-p, and why you chose those values
3. **Failure detection approach** — what you log, what you check automatically, and where you require human review

Bring it to the next session for peer review.


In [ ]:
take_home_brief = """
## Take-Home Technical Brief

**Integration:**
[Name of integration / system]

**1. Context Window Strategy:**
[Describe how you manage input length and position critical information]

**2. Parameter Configuration Rationale:**
[List: temperature=X because..., top-p=X because...]

**3. Failure Detection Approach:**
[Describe what you log, what automated checks you run, and where human review is required]
"""

print(take_home_brief)


---
*Session 1 Lab Notebook · Technical Intro to Generative AI · McKinsey Internal Training*
